# Data Deduplication Lab - Getting Started

Welcome to the Data Deduplication Lab. This notebook prepares your **Cloudera AI Workbench** session for the Phase 1 exercises.

## Learning Objectives

By the end of this notebook, you will:
- Install project Python dependencies from `requirements.txt`
- Create a **local-mode** Spark session in Cloudera AI Workbench (`local[*]`)
- Read sample customer data from the project `data/` directory
- Copy that local file to HDFS with **`hdfs dfs`** CLI, then verify it
- Inspect duplicate patterns on `name` + `email`

## What this notebook does

1. **Dependencies** — install packages from `../requirements.txt`
2. **Spark setup** — create a local Spark session (local files only; no forced `fs.defaultFS`)
3. **Local read** — load `../data/redundant_data.csv` via `file://`
4. **HDFS copy** — `hdfs dfs -put` the local file under `/tmp`, then verify with CLI
5. **Duplicate check** — profile uniqueness on key columns

## Prerequisites

- Cloudera AI Workbench session with PySpark available
- Project files including `use-case-phase-1/data/`
- Permission to read HDFS `/tmp` (or edit `HDFS_INPUT` to a path you can read)
- Basic Python familiarity

## Types of Deduplication (lab overview)

### Record-Level Deduplication (Exercise 1+)
- Removes duplicate **rows/records** within a dataset
- Example: two customer rows with the same name and email

**Next notebook after this setup:** `01_Basic_Deduplication.ipynb`


## 0. Install Project Requirements

Install packages from `use-case-phase-1/requirements.txt` into this session (includes `pyspark` if it is not already on the kernel path). Re-run this cell after restarting the kernel if imports fail.


In [ ]:
from pathlib import Path
import importlib
import sys
import subprocess

REQ_FILE = Path("../requirements.txt").resolve()
assert REQ_FILE.is_file(), f"Missing requirements file: {REQ_FILE}"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])

# Ensure a fresh import after install (handles kernels that started without pyspark)
importlib.invalidate_caches()
import pyspark

print(f"✓ Installed requirements from: {REQ_FILE}")
print(f"✓ pyspark {pyspark.__version__} available")


## 1. Create Local Spark Session

This lab uses **local mode** (`local[*]`). Do **not** set `spark.hadoop.fs.defaultFS` to `hdfs://ns1` on pip-installed PySpark — without cluster `hdfs-site.xml` that nameservice does not resolve and even local reads fail.

Use Spark for local `file://` data. Use the **`hdfs` CLI** (which has CAI Hadoop config) for HDFS put/ls.


In [ ]:
from pyspark.sql import SparkSession

# Logical HDFS path used by hdfs CLI (cluster defaultFS / nameservice comes from CAI Hadoop config)
HDFS_DIR = "/tmp/cdp_user_demo/phase1"
HDFS_PATH = f"{HDFS_DIR}/redundant_data.csv"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DeduplicationLab_GettingStarted")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"HDFS path:     {HDFS_PATH}")
print("✓ Spark session created successfully")
print("⚠ If you previously set fs.defaultFS=hdfs://ns1, restart the kernel before continuing.")


## 2. Read Local Sample Data

Load the project CSV from `use-case-phase-1/data/`. Schema: `id`, `name`, `email`, `address`.

For a larger workload later, switch the filename to `redundant_data_large.csv`.


In [ ]:
from pathlib import Path

LOCAL_DATA_DIR = Path("../data").resolve()
LOCAL_INPUT = LOCAL_DATA_DIR / "redundant_data.csv"
# Optional scale-up file:
# LOCAL_INPUT = LOCAL_DATA_DIR / "redundant_data_large.csv"

assert LOCAL_INPUT.is_file(), f"Missing local file: {LOCAL_INPUT}"

# Explicit file:// so Spark does not try to interpret the path via HDFS
LOCAL_URI = LOCAL_INPUT.resolve().as_uri()

df_local = spark.read.csv(LOCAL_URI, header=True, inferSchema=True)

print(f"✓ Loaded local file: {LOCAL_URI}")
print(f"Total records: {df_local.count():,}")
print(f"Columns: {', '.join(df_local.columns)}")
print("\nPreview:")
df_local.show(10, truncate=False)
df_local.printSchema()


## 3. Copy Local Data to HDFS (`hdfs dfs`)

Use the CAI **`hdfs`** CLI (not Spark) so the cluster Hadoop config / nameservice is applied correctly.


In [ ]:
import shutil
import subprocess
from pathlib import Path

hdfs_bin = shutil.which("hdfs") or shutil.which("hadoop")
assert hdfs_bin, "Neither 'hdfs' nor 'hadoop' found on PATH — run this in a CAI Workbench session with Hadoop client tools."

def hdfs_cmd(*args: str) -> None:
    """Run an HDFS CLI command (hdfs dfs ... or hadoop fs ...)."""
    if Path(hdfs_bin).name == "hadoop":
        cmd = [hdfs_bin, "fs", *args]
    else:
        cmd = [hdfs_bin, "dfs", *args]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

hdfs_cmd("-mkdir", "-p", HDFS_DIR)
hdfs_cmd("-put", "-f", str(LOCAL_INPUT), HDFS_PATH)

print(f"✓ Copied to HDFS: {HDFS_PATH}")


## 4. Verify the HDFS File (CLI)

Confirm with `hdfs dfs -ls` / `-cat`. Pip PySpark often cannot resolve the cluster nameservice, so verification stays on the CLI here.


In [ ]:
hdfs_cmd("-ls", "-h", HDFS_PATH)

cat_cmd = (
    [hdfs_bin, "fs", "-cat", HDFS_PATH]
    if Path(hdfs_bin).name == "hadoop"
    else [hdfs_bin, "dfs", "-cat", HDFS_PATH]
)
preview = subprocess.check_output(cat_cmd, text=True)
print("--- first lines ---")
for line in preview.splitlines()[:6]:
    print(line)

print(f"\n✓ Verified HDFS file: {HDFS_PATH}")


## 5. Quick Duplicate Check (Local Data)

Profile uniqueness on `name` + `email` in the local dataset — the same keys used in Exercise 1.


In [ ]:
KEY_COLS = ["name", "email"]

total_count = df_local.count()
unique_count = df_local.select(*KEY_COLS).distinct().count()
duplicates_count = total_count - unique_count
duplicate_rate = (duplicates_count / total_count * 100) if total_count else 0

print(f"Total records: {total_count:,}")
print(f"Unique records (by name+email): {unique_count:,}")
print(f"Duplicate records: {duplicates_count:,}")
print(f"Duplicate rate: {duplicate_rate:.2f}%")
print(f"\nLocal path for Exercise 1:\n  {LOCAL_INPUT}")
print(f"HDFS path (CLI):\n  {HDFS_PATH}")


## Next Steps

Setup is complete. Exercise 1 defaults to the **local** CSV:

```text
../data/redundant_data.csv
```

HDFS copy (via `hdfs dfs -put`):

```text
/tmp/cdp_user_demo/phase1/redundant_data.csv
```

1. **Exercise 1**: Basic Deduplication — `01_Basic_Deduplication.ipynb`
2. **Exercise 2**: Iceberg REST Catalog — `02_Iceberg_REST_Catalog.ipynb`

## Cleanup

**Restart the kernel** if you previously created a Spark session with `fs.defaultFS=hdfs://ns1`, then re-run from the top. Stop the Spark session when finished:


In [ ]:
spark.stop()
print("✓ Spark session stopped")
